# Classification Intel (Kaggle) — notebook sans Flask

Pipeline **PyTorch** et **TensorFlow** avec le **même prétraitement** (ImageNet) et des augmentations **alignées** autant que possible.

**À faire :** remplacer `FIRSTNAME` à l’étape 0, puis exécuter les cellules dans l’ordre.

## Étape 0 — Imports, constantes et vérification du dataset (exemple)

On fixe les chemins Kaggle, les **hyperparamètres communs** aux deux frameworks, et on vérifie que les dossiers / classes existent.

In [ ]:
import os
import random
import json
from pathlib import Path

import numpy as np

# --- Constantes partagées (même logique PyTorch / TensorFlow) ---
SEED = 42
FIRSTNAME = "your_firstname"  # <-- remplacer par ton prénom (fichiers exportés)

DATA_ROOT = "/kaggle/input/intel-image-classification"
TRAIN_DIR = os.path.join(DATA_ROOT, "seg_train", "seg_train")
TEST_DIR = os.path.join(DATA_ROOT, "seg_test", "seg_test")
OUT_DIR = "/kaggle/working/artifacts"

CLASS_NAMES = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
NUM_CLASSES = len(CLASS_NAMES)

IMAGE_SIZE = 150
BATCH_SIZE = 32
EPOCHS = 15
LR = 1e-3

# Normalisation ImageNet (identique à torchvision.transforms.Normalize)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# Augmentations : mêmes ordres de grandeur PT / TF
AUG_ROTATION_DEG = 10
# PyTorch ColorJitter(0.1,0.1,0.1) ~ léger bruit couleur ; TF approxime avec brightness/contrast
AUG_BRIGHTNESS = 0.1
AUG_CONTRAST = 0.1
AUG_ZOOM = 0.1  # pas d’équivalent direct dans le bloc PT ; léger zoom TF seulement

# --- Vérification des chemins (exemple diagnostic) ---
assert os.path.isdir(TRAIN_DIR), f"Dossier train introuvable : {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR), f"Dossier test introuvable : {TEST_DIR}"

found_train = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
print("Classes trouvées (train) :", found_train)
assert found_train == sorted(CLASS_NAMES), "Ordre ou noms de classes différents du jeu Intel attendu."

n_train = sum(len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in CLASS_NAMES)
n_test = sum(len(os.listdir(os.path.join(TEST_DIR, c))) for c in CLASS_NAMES)
print(f"Exemple comptage — images train: {n_train}, test: {n_test}")
print("Dossier sortie artefacts:", OUT_DIR)

## Étape 1 — Reproductibilité

Graines et options **déterministes** pour rapprocher les runs (utile pour rapport et déploiement).

In [ ]:
import tensorflow as tf
import torch


def set_global_seed(seed: int = 42) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    tf.random.set_seed(seed)


set_global_seed(SEED)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print("Seed appliquée:", SEED)
print("PyTorch CUDA:", torch.cuda.is_available())
print("TensorFlow GPU:", bool(tf.config.list_physical_devices("GPU")))

## Étape 2 — Modèle PyTorch et chargeurs de données

`Resize` → augmentations → `ToTensor` → **Normalize(ImageNet)**.  
Même découpage **80 % train / 20 % val** que pour TensorFlow (sur les batches du flux train).

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms


def build_torch_dataloaders(train_dir, test_dir, image_size, batch_size, seed):
    train_tfms = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(AUG_ROTATION_DEG),
        transforms.ColorJitter(
            brightness=AUG_BRIGHTNESS,
            contrast=AUG_CONTRAST,
            saturation=AUG_BRIGHTNESS,
        ),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    eval_tfms = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    full_train = datasets.ImageFolder(train_dir, transform=train_tfms)
    val_size = int(0.2 * len(full_train))
    train_size = len(full_train) - val_size
    gen = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(full_train, [train_size, val_size], generator=gen)
    val_ds.dataset = datasets.ImageFolder(train_dir, transform=eval_tfms)

    test_ds = datasets.ImageFolder(test_dir, transform=eval_tfms)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader


class TorchCNN(nn.Module):
    """CNN custom (architecture distincte du bloc TF, mais même taille d’entrée)."""

    def __init__(self, num_classes: int = 6):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 18 * 18, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

## Étape 3 — Modèle TensorFlow et datasets (cohérent avec l’étape 2)

Même chaîne : **`[0,1]` puis `(x - mean) / std`**. 
Augmentations appliquées **avant** la normalisation (comme l’esprit PyTorch : géométrie / couleur sur l’image réelle).

Rotation TF : `factor = 10° / 360°` pour équivaloir à `RandomRotation(10)` (degrés).

**Note TF** : le message rouge `use_unbounded_threadpool` / `MapDataset` est un avertissement de version TensorFlow (souvent sans impact). Si l’accuracy TF restait ~17 % (hasard sur 6 classes), c’était en général dû au **mauvais `value_range`** des couches couleur sur des pixels `[0,1]` — corrigé ci‑dessous avec `value_range=(0., 1.)`.

In [ ]:
def build_tf_datasets(train_dir, test_dir, image_size, batch_size, seed):
    # Split officiel 80/20 (meme seed) — plus fiable que take/skip sur le flux batché
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        label_mode="int",
        image_size=(image_size, image_size),
        batch_size=batch_size,
        validation_split=0.2,
        subset="training",
        seed=seed,
        shuffle=True,
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        label_mode="int",
        image_size=(image_size, image_size),
        batch_size=batch_size,
        validation_split=0.2,
        subset="validation",
        seed=seed,
        shuffle=True,
    )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        test_dir,
        label_mode="int",
        image_size=(image_size, image_size),
        batch_size=batch_size,
        shuffle=False,
    )

    imagenet_mean = tf.constant(IMAGENET_MEAN, dtype=tf.float32)
    imagenet_std = tf.constant(IMAGENET_STD, dtype=tf.float32)

    # Images ici sont en float [0,1] : imposer value_range evite un mauvais scaling (acc ~ chance)
    vr = (0.0, 1.0)
    rot_factor = float(AUG_ROTATION_DEG) / 360.0
    aug = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal", seed=seed),
        tf.keras.layers.RandomRotation(rot_factor, fill_mode="reflect", seed=seed),
        tf.keras.layers.RandomBrightness(AUG_BRIGHTNESS, value_range=vr, seed=seed),
        tf.keras.layers.RandomContrast(AUG_CONTRAST, value_range=vr, seed=seed),
        tf.keras.layers.RandomZoom(AUG_ZOOM, AUG_ZOOM, fill_mode="reflect", seed=seed),
    ])

    def prep_train(x, y):
        x = tf.cast(x, tf.float32) / 255.0
        x = aug(x, training=True)
        x = (x - imagenet_mean) / imagenet_std
        return x, y

    def prep_eval(x, y):
        x = tf.cast(x, tf.float32) / 255.0
        x = (x - imagenet_mean) / imagenet_std
        return x, y

    # num_parallel_calls=1 limite souvent l’avertissement MapDataset / threadpool sur certaines versions TF
    auto = tf.data.AUTOTUNE
    train_ds = train_ds.map(prep_train, num_parallel_calls=1).prefetch(auto)
    val_ds = val_ds.map(prep_eval, num_parallel_calls=1).prefetch(auto)
    test_ds = test_ds.map(prep_eval, num_parallel_calls=1).prefetch(auto)
    return train_ds, val_ds, test_ds


def build_tf_model(num_classes: int, image_size: int, lr: float):
    inputs = tf.keras.Input(shape=(image_size, image_size, 3))
    x = tf.keras.layers.Conv2D(32, 3, padding="same")(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.MaxPool2D()(x)
    x = tf.keras.layers.Conv2D(64, 3, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.MaxPool2D()(x)
    x = tf.keras.layers.Conv2D(128, 3, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.MaxPool2D()(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

## Étape 4 — Entraînement, métriques et sauvegarde des artefacts

- **PyTorch** : `ReduceLROnPlateau` sur la val accuracy, checkpoint meilleur `.pth`.
- **TensorFlow** : `ModelCheckpoint`, `EarlyStopping`, `ReduceLROnPlateau`.
- Métadonnées communes exportées dans `*_meta.json` (classes, taille d’image, seed, framework).

In [ ]:
def train_pytorch(firstname=FIRSTNAME):
    set_global_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_loader, val_loader, test_loader = build_torch_dataloaders(
        TRAIN_DIR, TEST_DIR, IMAGE_SIZE, BATCH_SIZE, SEED
    )

    model = TorchCNN(NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )

    best_acc = -1.0
    best_path = os.path.join(OUT_DIR, f"{firstname}_model.pth")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total, correct, run_loss = 0, 0, 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            run_loss += loss.item() * y.size(0)
            total += y.size(0)
            correct += (logits.argmax(1) == y).sum().item()

        model.eval()
        v_total, v_correct = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                v_total += y.size(0)
                v_correct += (logits.argmax(1) == y).sum().item()

        tr_acc = correct / total
        va_acc = v_correct / v_total
        scheduler.step(va_acc)
        print(f"[PT] Epoch {epoch}/{EPOCHS} train_acc={tr_acc:.4f} val_acc={va_acc:.4f}")

        if va_acc > best_acc:
            best_acc = va_acc
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "class_names": CLASS_NAMES,
                    "image_size": IMAGE_SIZE,
                    "imagenet_mean": list(IMAGENET_MEAN),
                    "imagenet_std": list(IMAGENET_STD),
                    "seed": SEED,
                    "framework": "pytorch",
                },
                best_path,
            )

    model.load_state_dict(torch.load(best_path, map_location=device)["model_state"])
    model.eval()
    te_correct, te_total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            te_correct += (model(x).argmax(1) == y).sum().item()
            te_total += y.size(0)
    print(f"[PT] Test accuracy: {te_correct / te_total:.4f}")
    print("Saved:", best_path)
    meta_pt = {
        "class_names": CLASS_NAMES,
        "image_size": IMAGE_SIZE,
        "imagenet_mean": list(IMAGENET_MEAN),
        "imagenet_std": list(IMAGENET_STD),
        "seed": SEED,
        "framework": "pytorch",
    }
    with open(os.path.join(OUT_DIR, f"{firstname}_meta_pytorch.json"), "w", encoding="utf-8") as f:
        json.dump(meta_pt, f, indent=2)
    with open(os.path.join(OUT_DIR, f"{firstname}_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta_pt, f, indent=2)


def train_tensorflow(firstname=FIRSTNAME):
    set_global_seed(SEED)
    train_ds, val_ds, test_ds = build_tf_datasets(
        TRAIN_DIR, TEST_DIR, IMAGE_SIZE, BATCH_SIZE, SEED
    )
    model = build_tf_model(NUM_CLASSES, IMAGE_SIZE, LR)

    best_path = os.path.join(OUT_DIR, f"{firstname}_model.keras")
    cbs = [
        tf.keras.callbacks.ModelCheckpoint(
            best_path, monitor="val_accuracy", save_best_only=True, verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=6, min_delta=1e-4, restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_accuracy", factor=0.5, patience=2, min_lr=1e-6, verbose=1
        ),
    ]

    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=cbs, verbose=1)
    test_metrics = model.evaluate(test_ds, verbose=0)
    print(f"[TF] Test loss={test_metrics[0]:.4f} acc={test_metrics[1]:.4f}")
    print("Saved (meilleur checkpoint):", best_path)

    meta = {
        "class_names": CLASS_NAMES,
        "image_size": IMAGE_SIZE,
        "imagenet_mean": list(IMAGENET_MEAN),
        "imagenet_std": list(IMAGENET_STD),
        "seed": SEED,
        "framework": "tensorflow",
    }
    with open(os.path.join(OUT_DIR, f"{firstname}_meta_tensorflow.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    # Copie pratique pour outils qui attendent un seul fichier meta :
    with open(os.path.join(OUT_DIR, f"{firstname}_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)


def run(framework: str = "tensorflow", firstname: str = FIRSTNAME):
    if framework == "pytorch":
        train_pytorch(firstname=firstname)
    elif framework == "tensorflow":
        train_tensorflow(firstname=firstname)
    else:
        raise ValueError("framework doit être 'pytorch' ou 'tensorflow'")

## Étape 5 — Exécution : choisir PyTorch ou TensorFlow

Décommente **une** ligne `run(...)` puis exécute la cellule. Les fichiers vont dans `OUT_DIR`.

In [ ]:
# run("pytorch", firstname=FIRSTNAME)
# run("tensorflow", firstname=FIRSTNAME)

print("Décommente run(\"pytorch\"...) ou run(\"tensorflow\"...) ci-dessus.")
print("Artefacts:", OUT_DIR)

## Étape 6 — Vérification des fichiers exportés (optionnel)

Après entraînement : `prenom_model.pth`, `prenom_model.keras`, `prenom_meta_pytorch.json`, `prenom_meta_tensorflow.json`, et `prenom_meta.json` (copie du dernier meta écrit — relance TensorFlow en dernier si tu utilises un seul fichier côté Flask).

In [ ]:
from pathlib import Path as _Path

for p in sorted(_Path(OUT_DIR).glob("*")):
    print(p.name, f"({p.stat().st_size} bytes)")